# Оптимизация параметров застройки с помощью LLM  агентов
1. Устновить ollama с помощью brew (brew install ollama)
2. Запустить ollama сервер (ollama serve)
3. Убедитесь, что сервер расположен на порте 11434
4. Скачайте модель (ollama pull llama3)
5. После установки проверьте работоспособность (ollama run llama3)

In [ ]:
from catboost import CatBoostRegressor
import pandas as pd
from urbanomy.methods.land_value_modeling import LandPriceEstimator

model = CatBoostRegressor()
model.load_model('/Users/andreyfeduhin/Downloads/Urbanomy/examples/land_value_modeling/catboost_model.cbm')  # модель на лог-цене

In [ ]:
import pandas as pd

scenario_blocks = pd.read_pickle('/Users/andreyfeduhin/Urbanomy-data/data/leningrad_oblast/shlisselburg/scenario_blocks.pickle') #or baseline_blocks
context_blocks = pd.read_pickle('/Users/andreyfeduhin/Urbanomy-data/data/leningrad_oblast/shlisselburg/context_blocks.pickle')


In [ ]:
from urbanomy.methods.land_value_modeling.land_data_preparation import LandDataPreparator

preparator = LandDataPreparator(
    scenario_blocks_source=scenario_blocks,
    context_blocks_source=context_blocks,
    predict_project_only=False, 
)

prepared_blocks = preparator.prepare()
prepared_blocks.head()

In [ ]:
feature_cols = [
'residential','business','recreation','industrial','transport','special',
'agriculture','land_use','share','footprint_area','build_floor_area',
'living_area','non_living_area','population','site_area','fsi','gsi',
'mxi','l','morphotype','area_accessibility'
]
cat_features = ['land_use', 'morphotype']
numeric_feats = [c for c in feature_cols if c not in cat_features]

In [ ]:
from urbanomy.methods.land_value_modeling import (
    ScenarioTEPModifier,
    plot_scenario_impact,
)

blocks_before = prepared_blocks.copy()
changes = {
    'land_use': 'LandUse.RESIDENTIAL',
    'share': 0.95,
    'footprint_area': 49039.24,
    'build_floor_area': 230279.19,
    'living_area': 155896.85,
    'population': 5168
}
target_idx = 62

modifier = ScenarioTEPModifier(blocks_before)
blocks_after = modifier.apply(target_idx, changes)

scenario_result = plot_scenario_impact(
    blocks_before=blocks_before,
    blocks_after=blocks_after,
    model=model,
    orig_features=numeric_feats+cat_features,
    categorical_features=cat_features,
    target_idx=target_idx,
    figsize=(25, 35),
)



In [ ]:
from urbanomy.methods.land_value_modeling.scenario_modification import GeneticOptimizer
blocks_before["site_area"].iloc[62]

In [ ]:
site_area = float(blocks_before["site_area"].iloc[62])
floors_neighborhood_avg = blocks_before['build_floor_area'].mean() 
optimizer = GeneticOptimizer(
    blocks_before=blocks_before,
    target_idx=target_idx,
    model=model,
    estimator_kwargs={
        "orig_features": numeric_feats+cat_features,
        "categorical_features": cat_features,
    },
    constraints = {
    # Площадь пятна застройки
    "footprint_area": {
        "type": "float",
        "min": 0.0,
        "max": 0.8 * site_area,
    },
    # Площадь всех этажей
    "build_floor_area": {
        "type": "float",
        "min": site_area,
        "max": floors_neighborhood_avg*2,  
    },
    # Жилая площадь
    "living_area": {
        "type": "float",
        "min": 0.0,
        "max": floors_neighborhood_avg*2,  
    },
    # land_use 
    "residential": {"type": "float", "min": 0.0, "max": 1.0},
        "business":    {"type": "float", "min": 0.0, "max": 1.0},
        "recreation":  {"type": "float", "min": 0.0, "max": 1.0},
        "industrial":  {"type": "float", "min": 0.0, "max": 1.0},
        "transport":   {"type": "float", "min": 0.0, "max": 1.0},
        "special":     {"type": "float", "min": 0.0, "max": 1.0},
        "agriculture": {"type": "float", "min": 0.0, "max": 1.0}}, random_state=42)

In [ ]:
best_score,best_genome = optimizer.optimize(generations=10, population_size=20)


In [ ]:
best_genome

In [ ]:
changable_params = ['footprint_area', 'build_floor_area', 'living_area', 'residential', 'business', 'recreation', 'industrial', 'transport', 'special', 'agriculture']

In [ ]:
genome_for_llm = blocks_before.loc[62][changable_params].to_dict()
genome_for_llm

In [ ]:
from urbanomy.methods.land_value_modeling.llm_agents import OllamaLLM, CityAdministrationAgent, ResidentsAgent, InvestorsAgent, DeveloperAggregator, run_multiagent_loop

llm = OllamaLLM(model="llama3", temperature=0.7)

print(llm.generate("Скажи одним предложением, что такое город"))

In [ ]:
agents = [
    CityAdministrationAgent("city", llm),
    ResidentsAgent("residents", llm),
    InvestorsAgent("investors", llm),
]

In [ ]:
aggregator = DeveloperAggregator(
    llm=llm,
    agents=agents
)

In [ ]:
# 1. Стартовый геном 
genome_for_llm = blocks_before.loc[62][changable_params].to_dict()

# 2. Multi-agent переговоры
genome = run_multiagent_loop(
    initial_genome=genome_for_llm,
    aggregator=aggregator,
    n_steps=3
)

In [ ]:
genome_for_llm

In [ ]:
genome = optimizer._repair_genome(genome)
genome

In [ ]:
optimizer.plot_best(genome)